In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import FunctionTransformer
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                              precision_score, recall_score, f1_score, roc_curve)
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold

In [46]:
import warnings
from sklearn.exceptions import DataConversionWarning

warnings.filterwarnings('ignore', message='Found unknown categories')

In [23]:
df = pd.read_csv("../data/cleaned/cleaned_telco_churn.csv")

In [24]:
df.head()

,customer_i_d,gender,age,married,number_of_dependents,city,zip_code,latitude,longitude,number_of_referrals,...,monthly_charge,total_charges,total_refunds,total_extra_data_charges,total_long_distance_charges,total_revenue,customer_status,churn_category,churn_reason,population
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.827662,-118.999073,2,...,65.6,593.30,0.00,0,381.51,974.81,Stayed,not_churned,not_churned,4498
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.162515,-118.203869,0,...,-4.0,542.40,38.33,10,96.21,610.28,Stayed,not_churned,not_churned,31297
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.645672,-117.922613,0,...,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices,62069
3,0011-IGKFF,Male,78,Yes,0,Martinez,94553,38.014457,-122.115432,1,...,98.0,1237.85,0.00,0,361.66,1599.51,Churned,Dissatisfaction,Product dissatisfaction,46677
4,0013-EXCHZ,Female,75,Yes,0,Camarillo,93010,34.227846,-119.079903,3,...,83.9,267.40,0.00,0,22.14,289.54,Churned,Dissatisfaction,Network reliability,42853


In [25]:
# Filter out "Joined" customers since their outcome (stay or churn) isn't determined yet
# We only keep customers whose status is either "Stayed" or "Churned"
df_model = df[df['customer_status'] != 'Joined'].copy()

In [26]:
df_model['churn_label'] = df_model['customer_status'].map({'Stayed': 0, 'Churned': 1})

In [27]:
print(df_model['customer_status'].value_counts())
print(df_model['churn_label'].value_counts())
print(df_model.shape)

customer_status
Stayed     4720
Churned    1869
Name: count, dtype: int64
churn_label
0    4720
1    1869
Name: count, dtype: int64
(6589, 40)


In [28]:
# Columns to drop:
# - customer_status: replaced by our new churn_label, no longer needed
# - churn_category, churn_reason: these are only known AFTER a customer churns (data leakage)
# - customer_i_d: just a unique identifier, has no predictive pattern
# - zip_code, latitude, longitude: too specific/granular, easily overfit, and city/population already give general location context

cols_to_drop = ['customer_status', 'churn_category', 'churn_reason', 
                'customer_i_d', 'zip_code', 'latitude', 'longitude']

df_model = df_model.drop(columns=cols_to_drop)

print(df_model.columns.tolist())
print(df_model.shape)

['gender', 'age', 'married', 'number_of_dependents', 'city', 'number_of_referrals', 'tenure_in_months', 'offer', 'phone_service', 'avg_monthly_long_distance_charges', 'multiple_lines', 'internet_service', 'internet_type', 'avg_monthly_g_b_download', 'online_security', 'online_backup', 'device_protection_plan', 'premium_tech_support', 'streaming_t_v', 'streaming_movies', 'streaming_music', 'unlimited_data', 'contract', 'paperless_billing', 'payment_method', 'monthly_charge', 'total_charges', 'total_refunds', 'total_extra_data_charges', 'total_long_distance_charges', 'total_revenue', 'population', 'churn_label']
(6589, 33)


## Feature Engineering Create New Ratio-Based Features

In [29]:
# 1. Average revenue per month of tenure
# Helps compare customers fairly regardless of how long they've been around
df_model['revenue_per_month'] = df_model['total_revenue'] / df_model['tenure_in_months']

# 2. Refund ratio — how much of what they were charged came back as a refund
# A high refund ratio might signal dissatisfaction
df_model['refund_ratio'] = df_model['total_refunds'] / df_model['total_charges']

# 3. Extra data charge ratio — how much of their revenue came from "extra" overage charges
df_model['extra_data_ratio'] = df_model['total_extra_data_charges'] / df_model['total_revenue']

# 4. Long distance charge ratio — how much of their revenue came from long distance calls
df_model['long_distance_ratio'] = df_model['total_long_distance_charges'] / df_model['total_revenue']

# 5. Tenure buckets — group customers into "new," "established," and "loyal" categories
df_model['tenure_group'] = pd.cut(
    df_model['tenure_in_months'],
    bins=[0, 12, 24, 48, 100],
    labels=['0-12mo', '12-24mo', '24-48mo', '48mo+']
)

# Replace any division-by-zero errors (inf) or missing values with 0
# Only applied to the numeric ratio columns — NOT the whole dataframe,
# since tenure_group is categorical and can't be filled with 0
ratio_cols = ['revenue_per_month', 'refund_ratio', 'extra_data_ratio', 'long_distance_ratio']
df_model[ratio_cols] = df_model[ratio_cols].replace([np.inf, -np.inf], 0).fillna(0)

# Quick check
print(df_model[['revenue_per_month', 'refund_ratio', 'extra_data_ratio', 
                 'long_distance_ratio', 'tenure_group']].head())

   revenue_per_month  refund_ratio  extra_data_ratio  long_distance_ratio  \
0         108.312222      0.000000          0.000000             0.391369   
1          67.808889      0.070667          0.016386             0.157649   
2         103.862500      0.000000          0.000000             0.323986   
3         123.039231      0.000000          0.000000             0.226107   
4          96.513333      0.000000          0.000000             0.076466   

  tenure_group  
0       0-12mo  
1       0-12mo  
2       0-12mo  
3      12-24mo  
4       0-12mo  


In [30]:
X = df_model.drop(columns=['churn_label'])
y = df_model['churn_label']

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [32]:
numeric_cols = ['age', 'number_of_dependents', 'number_of_referrals', 'tenure_in_months',
                'avg_monthly_long_distance_charges', 'avg_monthly_g_b_download',
                'monthly_charge', 'total_charges', 'total_refunds', 
                'total_extra_data_charges', 'total_long_distance_charges', 
                'total_revenue', 'population', 'revenue_per_month', 
                'refund_ratio', 'extra_data_ratio', 'long_distance_ratio']

categorical_cols = ['gender', 'married', 'offer', 'phone_service', 'multiple_lines',
                     'internet_service', 'internet_type', 'online_security', 
                     'online_backup', 'device_protection_plan', 'premium_tech_support',
                     'streaming_t_v', 'streaming_movies', 'streaming_music', 
                     'unlimited_data', 'contract', 'paperless_billing', 
                     'payment_method', 'tenure_group', 'city']

In [33]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols)
])

In [34]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

C:\Users\Swayam B Solanki\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\preprocessing\_encoders.py:241: UserWarning: Found unknown categories in columns [19] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [35]:
print(X_train_processed.shape)
print(X_test_processed.shape)

(5271, 1146)
(1318, 1146)


In [36]:
# Build the full pipeline: preprocessing Logistic Regression
logreg_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Define the hyperparameter grid to search over
logreg_param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100],           
    'classifier__penalty': ['l1', 'l2'],                 
    'classifier__solver': ['liblinear']                 
}

In [37]:
# Set up cross-validation strategy 
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

logreg_grid = GridSearchCV(
    estimator=logreg_pipeline,
    param_grid=logreg_param_grid,
    scoring='roc_auc',
    cv=cv_strategy,
    n_jobs=-1,         
    verbose=1
)

In [38]:
logreg_grid.fit(X_train, y_train)

# Best parameters found
print("Best parameters:", logreg_grid.best_params_)
print("Best cross-validation ROC-AUC:", round(logreg_grid.best_score_, 4))

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters: {'classifier__C': 0.1, 'classifier__penalty': 'l1', 'classifier__solver': 'liblinear'}
Best cross-validation ROC-AUC: 0.9179


In [19]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    print(f"\n===== {name} =====")
    print(classification_report(y_test, y_pred, target_names=['Stayed', 'Churned']))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("ROC-AUC Score:", round(roc_auc_score(y_test, y_proba), 4))
    
    return {
        'model_name': name,
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba)
    }


if 'results' not in dir():
    results = []


best_logreg = logreg_grid.best_estimator_
logreg_results = evaluate_model("Logistic Regression", best_logreg, X_test, y_test)
results.append(logreg_results)


===== Logistic Regression =====
              precision    recall  f1-score   support

      Stayed       0.92      0.81      0.86       944
     Churned       0.63      0.83      0.72       374

    accuracy                           0.81      1318
   macro avg       0.78      0.82      0.79      1318
weighted avg       0.84      0.81      0.82      1318

Confusion Matrix:
 [[764 180]
 [ 65 309]]
ROC-AUC Score: 0.9112


C:\Users\Swayam B Solanki\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\preprocessing\_encoders.py:241: UserWarning: Found unknown categories in columns [19] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\Swayam B Solanki\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\preprocessing\_encoders.py:241: UserWarning: Found unknown categories in columns [19] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [39]:
from sklearn.preprocessing import FunctionTransformer

# Small helper to convert sparse matrix -> dense array
def to_dense(X):
    return X.toarray() if hasattr(X, "toarray") else X

# Build the pipeline: preprocessing -> convert to dense -> SMOTE -> Gaussian Naive Bayes
nb_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('to_dense', FunctionTransformer(to_dense)),
    ('smote', SMOTE(random_state=42)),
    ('classifier', GaussianNB())
])

nb_param_grid = {
    'classifier__var_smoothing': np.logspace(0, -9, num=10)
}

nb_grid = GridSearchCV(
    estimator=nb_pipeline,
    param_grid=nb_param_grid,
    scoring='roc_auc',
    cv=cv_strategy,
    n_jobs=-1,
    verbose=1
)

# Fit on training data only
nb_grid.fit(X_train, y_train)

print("Best parameters:", nb_grid.best_params_)
print("Best cross-validation ROC-AUC:", round(nb_grid.best_score_, 4))

# Evaluate on the test set
best_nb = nb_grid.best_estimator_
nb_results = evaluate_model("Naive Bayes", best_nb, X_test, y_test)
results.append(nb_results)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters: {'classifier__var_smoothing': np.float64(0.01)}
Best cross-validation ROC-AUC: 0.8824

===== Naive Bayes =====


C:\Users\Swayam B Solanki\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\preprocessing\_encoders.py:241: UserWarning: Found unknown categories in columns [19] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\Swayam B Solanki\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\preprocessing\_encoders.py:241: UserWarning: Found unknown categories in columns [19] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


              precision    recall  f1-score   support

      Stayed       0.95      0.66      0.78       944
     Churned       0.51      0.91      0.65       374

    accuracy                           0.73      1318
   macro avg       0.73      0.78      0.71      1318
weighted avg       0.82      0.73      0.74      1318

Confusion Matrix:
 [[620 324]
 [ 35 339]]
ROC-AUC Score: 0.8772


In [40]:
# Build the pipeline: preprocessing -> SMOTE -> Random Forest
rf_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Hyperparameter grid — kept reasonably small so GridSearchCV doesn't take forever
rf_param_grid = {
    'classifier__n_estimators': [100, 200, 300],          # number of trees
    'classifier__max_depth': [5, 10, 20, None],           # how deep each tree can grow
    'classifier__min_samples_split': [2, 5, 10],          # min samples needed to split a node
    'classifier__min_samples_leaf': [1, 2, 4],            # min samples allowed in a leaf
    'classifier__class_weight': [None, 'balanced']        # extra imbalance handling on top of SMOTE
}

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    scoring='roc_auc',
    cv=cv_strategy,
    n_jobs=-1,
    verbose=1
)

# Fit on training data only
rf_grid.fit(X_train, y_train)

print("Best parameters:", rf_grid.best_params_)
print("Best cross-validation ROC-AUC:", round(rf_grid.best_score_, 4))

# Evaluate on the test set
best_rf = rf_grid.best_estimator_
rf_results = evaluate_model("Random Forest", best_rf, X_test, y_test)
results.append(rf_results)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Best parameters: {'classifier__class_weight': None, 'classifier__max_depth': None, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 300}
Best cross-validation ROC-AUC: 0.9194


C:\Users\Swayam B Solanki\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\preprocessing\_encoders.py:241: UserWarning: Found unknown categories in columns [19] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\Swayam B Solanki\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\preprocessing\_encoders.py:241: UserWarning: Found unknown categories in columns [19] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(



===== Random Forest =====
              precision    recall  f1-score   support

      Stayed       0.89      0.93      0.91       944
     Churned       0.80      0.70      0.74       374

    accuracy                           0.86      1318
   macro avg       0.84      0.81      0.83      1318
weighted avg       0.86      0.86      0.86      1318

Confusion Matrix:
 [[880  64]
 [114 260]]
ROC-AUC Score: 0.9138


In [41]:
# Build the pipeline: preprocessing -> SMOTE -> XGBoost
xgb_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', XGBClassifier(random_state=42, eval_metric='logloss'))
])

# Hyperparameter grid — kept reasonable to avoid extremely long runtime
xgb_param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.1, 0.2],
    'classifier__subsample': [0.8, 1.0],
    'classifier__colsample_bytree': [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=xgb_param_grid,
    scoring='roc_auc',
    cv=cv_strategy,
    n_jobs=-1,
    verbose=1
)

# Fit on training data only
xgb_grid.fit(X_train, y_train)

print("Best parameters:", xgb_grid.best_params_)
print("Best cross-validation ROC-AUC:", round(xgb_grid.best_score_, 4))

# Evaluate on the test set
best_xgb = xgb_grid.best_estimator_
xgb_results = evaluate_model("XGBoost", best_xgb, X_test, y_test)
results.append(xgb_results)

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best parameters: {'classifier__colsample_bytree': 1.0, 'classifier__learning_rate': 0.1, 'classifier__max_depth': 5, 'classifier__n_estimators': 100, 'classifier__subsample': 1.0}
Best cross-validation ROC-AUC: 0.9407

===== XGBoost =====
              precision    recall  f1-score   support

      Stayed       0.88      0.93      0.91       944
     Churned       0.80      0.69      0.74       374

    accuracy                           0.86      1318
   macro avg       0.84      0.81      0.83      1318
weighted avg       0.86      0.86      0.86      1318

Confusion Matrix:
 [[880  64]
 [115 259]]
ROC-AUC Score: 0.9302


C:\Users\Swayam B Solanki\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\preprocessing\_encoders.py:241: UserWarning: Found unknown categories in columns [19] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\Swayam B Solanki\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\preprocessing\_encoders.py:241: UserWarning: Found unknown categories in columns [19] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [49]:
# Combine the three best-tuned models using soft voting (averages predicted probabilities)
voting_clf = VotingClassifier(
    estimators=[
        ('logreg', best_logreg),
        ('rf', best_rf),
        ('xgb', best_xgb)
    ],
    voting='soft'
)

# Fit on training data (each sub-model is already tuned, this just combines their outputs)
voting_clf.fit(X_train, y_train)

# Evaluate on the test set
voting_results = evaluate_model("Voting Ensemble (LogReg + RF + XGB)", voting_clf, X_test, y_test)
results.append(voting_results)


===== Voting Ensemble (LogReg + RF + XGB) =====
              precision    recall  f1-score   support

      Stayed       0.90      0.90      0.90       944
     Churned       0.74      0.75      0.75       374

    accuracy                           0.86      1318
   macro avg       0.82      0.82      0.82      1318
weighted avg       0.86      0.86      0.86      1318

Confusion Matrix:
 [[846  98]
 [ 93 281]]
ROC-AUC Score: 0.928


In [43]:
# Get feature names after preprocessing (numeric + one-hot encoded names)
feature_names = (numeric_cols + 
                  list(best_xgb.named_steps['preprocessor']
                       .named_transformers_['cat']
                       .get_feature_names_out(categorical_cols)))

# Extract importance scores from the tuned XGBoost model
importances = best_xgb.named_steps['classifier'].feature_importances_

# Build a sorted dataframe of feature importances
feat_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(feat_importance_df.head(15))

                        feature  importance
39            contract_Two Year    0.180269
38            contract_One Year    0.160089
28    internet_type_Fiber Optic    0.116187
3              tenure_in_months    0.050353
17                  gender_Male    0.040528
2           number_of_referrals    0.040131
1          number_of_dependents    0.032501
31            online_backup_Yes    0.029430
23                   offer_none    0.027415
41   payment_method_Credit Card    0.023302
32   device_protection_plan_Yes    0.022298
18                  married_Yes    0.021002
40        paperless_billing_Yes    0.019698
894              city_San Diego    0.019196
30          online_security_Yes    0.016655


In [47]:
from sklearn.metrics import precision_recall_curve


y_proba_xgb = best_xgb.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_xgb)

target_recall = 0.85

# Find ALL indices where recall meets the target, then take the LAST one
# (since recall decreases as threshold increases, the last valid index
# gives the highest threshold — and best precision — for this recall level)
valid_indices = np.where(recalls[:-1] >= target_recall)[0]  # exclude last element (no threshold attached)
best_idx = valid_indices[-1] if len(valid_indices) > 0 else 0

print(f"Threshold for ~{target_recall} recall: {thresholds[best_idx]:.3f}")
print(f"Precision at that threshold: {precisions[best_idx]:.3f}")
print(f"Actual recall at that threshold: {recalls[best_idx]:.3f}")

Threshold for ~0.85 recall: 0.256
Precision at that threshold: 0.665
Actual recall at that threshold: 0.850


In [52]:
import joblib

# Save the entire fitted pipeline (preprocessing + SMOTE + XGBoost) as one file
joblib.dump(final_xgb_pipeline, 'churn_model_pipeline.pkl')

# Also save the chosen decision threshold separately, since it's not part of the pipeline object
import json
with open('model_config.json', 'w') as f:
    json.dump({'decision_threshold': 0.256}, f)

print("Model and config saved successfully.")

NameError: name 'final_xgb_pipeline' is not defined